# Sistema de Recomendação de Filmes
**FBC (TF-IDF + RRF)  →  Motor Híbrido Cascade  →  FC (SVD)**

In [29]:
import import_ipynb
import engines

CBFEngine    = engines.CBFEngine
CFEngine     = engines.CFEngine
HybridEngine = engines.HybridEngine


## Inicialização dos Engines

In [30]:
cbf    = CBFEngine()
cf     = CFEngine()
hybrid = HybridEngine(cbf_engine=cbf, cf_engine=cf)
print("Engines prontos.")


Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19
vetor latente usuário: (610, 100)
vetor latente filme: (8536, 100)
vetor de vies de cada usuário: (610,)
vetor de vies de cada filme: (8536,)
Engines prontos.


## Demo — Recomendações para um Usuário

In [31]:
import pandas as pd

usuario_id = 73
threshold  = 4.0

df_train = pd.read_csv('data_spliting_hybrid/training.csv')
df_test  = pd.read_csv('data_spliting_hybrid/testing.csv')

perfil = df_train[
    (df_train['userId'] == usuario_id) & (df_train['rating'] >= threshold)
][['movieId', 'rating']].copy()

titles = cbf.df_cbf[['movie_id', 'title']].rename(columns={'movie_id': 'movieId'})
perfil = perfil.merge(titles, on='movieId', how='left')
perfil = perfil.sort_values('rating', ascending=False).reset_index(drop=True)
perfil.index = range(1, len(perfil) + 1)

print(f'Perfil do Usuário {usuario_id} — {len(perfil)} filmes bem avaliados no treino (nota >= {threshold}):')
print()
display(perfil[['title', 'rating']].rename(columns={'title': 'Filme', 'rating': 'Nota'}))


Perfil do Usuário 73 — 87 filmes bem avaliados no treino (nota >= 4.0):



,Filme,Nota
1,Inglourious Basterds (2009),5.0
2,Sherlock Holmes: A Game of Shadows (2011),5.0
3,Spider-Man 3 (2007),5.0
4,Watchmen (2009),5.0
5,Star Trek (2009),5.0
...,...,...
83,127 Hours (2010),4.0
84,Black Swan (2010),4.0
85,"Help, The (2011)",4.0
86,Moneyball (2011),4.0


In [32]:
rec = hybrid.recomendar(usuario_id, k_cbf=200, k=10, alpha=0.2)

# Filmes relevantes no teste (gabarito)
relevantes = set(
    df_test[(df_test['userId'] == usuario_id) & (df_test['rating'] >= threshold)]['movieId']
)

# Marca hits
rec['Hit'] = rec['movie_id'].apply(lambda x: 'HIT' if x in relevantes else '')
rec.index = range(1, len(rec) + 1)

# Métricas
hits      = rec['Hit'].eq('HIT').sum()
precision = hits / len(rec)
recall    = hits / len(relevantes) if relevantes else 0

print(f'Top 10 recomendações para o Usuário {usuario_id}:')
print()
display(rec[['title', 'score_hibrido', 'Hit']].rename(columns={
    'title': 'Filme',
    'score_hibrido': 'Score Híbrido',
    'Hit': 'Relevante?'
}))
print()
print(f'Filmes relevantes no teste (nota >= {threshold}): {len(relevantes)}')
print(f'Precision@10 : {precision:.2%}  ({hits} acertos em 10 recomendações)')
print(f'Recall@10    : {recall:.2%}  ({hits} de {len(relevantes)} filmes relevantes recuperados)')


Top 10 recomendações para o Usuário 73:



,Filme,Score Híbrido,Relevante?
1,Pulp Fiction (1994),0.994916,
2,Fight Club (1999),0.992996,
3,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),0.967720,
4,"Departed, The (2006)",0.966992,
5,WALL·E (2008),0.962810,
6,Spider-Man 2 (2004),0.957690,HIT
7,Blade Runner (1982),0.957482,
8,Ant-Man (2015),0.952998,
9,Thor: Ragnarok (2017),0.944241,
10,Eternal Sunshine of the Spotless Mind (2004),0.940331,



Filmes relevantes no teste (nota >= 4.0): 29
Precision@10 : 10.00%  (1 acertos em 10 recomendações)
Recall@10    : 3.45%  (1 de 29 filmes relevantes recuperados)
